# CYE AI Challenge (GDGoC NUST) — 03 · K6 + temporal neighbour smoothing
**Idea.** Rows are 10 s apart and shuffled, but consecutive readings are near-identical in all four signals. So a row's nearest neighbours *within the same set* (held-out days, or the test set) are mostly its neighbours **in time**. Shading and normal come in long continuous stretches, so averaging each row's predicted probabilities with its k nearest neighbours restores the time context the shuffle removed.
* **No labels used**: only the features of the rows being predicted (transductive, like normalising the test set).
* **No new model**: K6 stays exactly as it is; smoothing is a post-processing step (seconds).
* **Open-circuit guard re-applied after smoothing.**

**Decision rule (fixed in advance):** smoothing is used only if it beats plain K6 on **both** validation splits (LODO and 3-day blocks). k is chosen by the average of the two splits. A small shading-vs-normal bias is reported as a diagnostic and applied only if it gains ≥ 0.003 on both splits.

## 1 · Setup & load

In [19]:
import os, glob, time, json, hashlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220)

SEED, THREADS = 42, 4
CB_PARAMS = dict(iterations=300, learning_rate=0.1, depth=6, l2_leaf_reg=3.0, loss_function="MultiClass",
                 random_seed=SEED, thread_count=THREADS, verbose=0, allow_writing_files=False)
TIE = 0.003
np.random.seed(SEED)

LABELS = ["normal", "shading", "degradation", "short_circuit", "open_circuit"]
RARE = ["degradation", "short_circuit", "open_circuit"]
L2I = {l: i for i, l in enumerate(LABELS)}
RAW = ["dc_voltage", "dc_current", "irradiance", "panel_temp"]
plt.rcParams.update({"figure.dpi": 90, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
R, T0 = {}, time.time()

def find_data_dir():
    if os.environ.get("DATA_DIR"): return os.environ["DATA_DIR"]
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits: return os.path.dirname(hits[0])
    import kagglehub
    path = kagglehub.competition_download("gdgoc-cye-ai-2026")
    hits = glob.glob(os.path.join(path, "**", "train.csv"), recursive=True)
    return os.path.dirname(hits[0]) if hits else path

DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv")).sort_values("id").reset_index(drop=True)
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv")).sort_values("id").reset_index(drop=True)
sub   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
y_all, DAYS = train.label.map(L2I).to_numpy(), train.day.to_numpy()
print(train.shape, test.shape, DATA_DIR)

(44017, 7) (9617, 5) /kaggle/input/competitions/gdgoc-cye-ai-2026


## 2 · Physics features
Healthy-plant reference fitted (Huber regression) on the `normal` rows of the **training days of each fold**:
`v_exp = f(T, ln G, G)`, `i_exp = a·G + b·G·T`. The model only sees deviations from it:

| Feature | Formula | Reveals |
|---|---|---|
| `v_res` | V − v_exp | short circuit ≈ −35 V, degradation ≈ −15 V |
| `v_ratio` | V / v_exp | open circuit ≈ 0.5, short circuit ≈ 0.87 |
| `i_ratio` | I / i_exp | shading < 1, open circuit ≈ 0.5 |
| `r_app` | −v_res / I (Ω) | extra series resistance = degradation |
| `irradiance` | G | operating point (faults were only seen at high sun) |

In [20]:
Xv = lambda d: np.c_[d.panel_temp, np.log(d.irradiance.clip(lower=1)), d.irradiance / 1000]
Xi = lambda d: np.c_[d.irradiance, d.irradiance * d.panel_temp]

class HealthyPlant:
    def fit(self, d):
        n = d[d.label == "normal"]
        self.vm = HuberRegressor(max_iter=1000).fit(Xv(n), n.dc_voltage)
        self.im = HuberRegressor(max_iter=1000).fit(Xi(n), n.dc_current)
        self.tm = HuberRegressor(max_iter=1000).fit(np.c_[n.irradiance / 1000], n.panel_temp)
        return self
    def transform(self, d):
        f = d[RAW].copy()
        f["v_exp"]   = self.vm.predict(Xv(d))
        f["i_exp"]   = self.im.predict(Xi(d)).clip(min=0.05)
        f["v_res"]   = d.dc_voltage - f.v_exp
        f["v_ratio"] = d.dc_voltage / f.v_exp
        f["i_ratio"] = d.dc_current / f.i_exp
        f["r_app"]   = -f.v_res / d.dc_current.clip(lower=0.5)
        f["irr_c"]   = d.irradiance.clip(upper=IRR_CAP)
        f["t_excess"] = d.panel_temp - self.tm.predict(np.c_[d.irradiance / 1000])
        return f

IRR_CAP = 800
FEATS5   = ["irradiance", "v_res", "v_ratio", "i_ratio", "r_app"]
FEATS5C  = ["irr_c", "v_res", "v_ratio", "i_ratio", "r_app"]
FEATS5T  = FEATS5 + ["t_excess"]
FEATS5CT = FEATS5C + ["t_excess"]

def oc_guard(f):   # half voltage AND half current = one string lost
    return ((f.v_ratio < 0.65) & (f.i_ratio > 0.35) & (f.i_ratio < 0.70)).values

def apply_guard(p, f):
    g = oc_guard(f); p = p.copy()
    p[~g, L2I["open_circuit"]] = 0
    p[g] = 0; p[g, L2I["open_circuit"]] = 1
    return p / p.sum(1, keepdims=True).clip(min=1e-12)

def synth_degradation(tr, hp, rng, n=800, g_lo=780):
    # V drop = R * I, R drawn from the fold's real degradation rows, applied to healthy rows (G > g_lo) of ALL training days
    base = tr[(tr.label == "normal") & (tr.irradiance > g_lo)]
    r = hp.transform(tr).r_app.values[(tr.label == "degradation").values]
    if len(r) < 10 or len(base) == 0: return tr.iloc[:0]
    s = base.sample(n, replace=True, random_state=int(rng.integers(1e9))).copy()
    s["dc_voltage"] = s.dc_voltage - rng.choice(r, n) * rng.normal(1, .10, n) * s.dc_current
    s["label"] = "degradation"
    return s

def synth_short(tr, hp, rng, n=600, g_lo=600):
    # short circuit removes a fixed FRACTION of string voltage at any sun level; current ~unchanged
    base = tr[(tr.label == "normal") & (tr.irradiance > g_lo)]
    real = hp.transform(tr); m = (tr.label == "short_circuit").values
    if m.sum() < 10 or len(base) == 0: return tr.iloc[:0]
    k = rng.integers(0, m.sum(), n)
    s = base.sample(n, replace=True, random_state=int(rng.integers(1e9))).copy()
    s["dc_voltage"] = s.dc_voltage * real.v_ratio.values[m][k] * rng.normal(1, .005, n)
    s["dc_current"] = s.dc_current * real.i_ratio.values[m][k] * rng.normal(1, .005, n)
    s["label"] = "short_circuit"
    return s

def drift_aug(tr, rng, v_sd=3.0, i_sd=0.03):
    # one extra copy; each training day gets ONE voltage offset and ONE current scale (day-level calibration drift)
    s = tr.copy(); dd = sorted(s.day.unique())
    off = dict(zip(dd, rng.normal(0, v_sd, len(dd)))); sc = dict(zip(dd, rng.normal(1, i_sd, len(dd))))
    s["dc_voltage"] = s.dc_voltage + s.day.map(off); s["dc_current"] = s.dc_current * s.day.map(sc)
    return pd.concat([tr, s], ignore_index=True)

def catboost(): return CatBoostClassifier(**CB_PARAMS)

def proba5(m, X):
    p = np.zeros((len(X), 5)); p[:, np.asarray(m.classes_).astype(int).ravel()] = m.predict_proba(X); return p

def make_fp(feats, guard=True, drift=False, synth_deg=False, synth_wide=False, return_model=False):
    def fp(tr, va):
        rng = np.random.default_rng(SEED); hp = HealthyPlant().fit(tr); trd = tr
        if synth_deg and not synth_wide: trd = pd.concat([trd, synth_degradation(tr, hp, rng)], ignore_index=True)
        if synth_wide: trd = pd.concat([trd, synth_degradation(tr, hp, rng, g_lo=600), synth_short(tr, hp, rng, g_lo=600)], ignore_index=True)
        if drift: trd = drift_aug(trd, rng)
        m = catboost().fit(hp.transform(trd)[feats], trd.label.map(L2I))
        fv = hp.transform(va); p = proba5(m, fv[feats])
        if guard: p = apply_guard(p, fv)
        return (p, m) if return_model else p
    return fp

## 3 · Validation harness (Leave-One-Day-Out, pooled out-of-fold macro F1)

In [21]:
LODO   = [[d] for d in sorted(train.day.unique())]
BLOCK3 = [[0, 4, 11], [1, 5, 12], [2, 6, 8], [3, 7, 9], [10]]   # several unseen days at once, like test
BOARD, OOF = [], {}

def run_cv(name, fp, groups=LODO, kind="", note=""):
    t = time.time(); oof = np.zeros(len(train), int)
    for g in groups:
        va = np.isin(DAYS, g)
        out = fp(train[~va].reset_index(drop=True), train[va].reset_index(drop=True))
        oof[va] = out.argmax(1) if out.ndim == 2 else out
    f1 = f1_score(y_all, oof, average=None, labels=range(5))
    cv = "LODO" if groups is LODO else "BLOCK3"
    row = {"step": name, "cv": cv, "kind": kind, "macro_F1": round(f1.mean(), 4),
           **{l: round(v, 3) for l, v in zip(LABELS, f1)},
           "deg_recall_day8": round(float((oof[(DAYS == 8) & (y_all == 2)] == 2).mean()), 3),
           "false_rare_on_clean_days": int((np.isin(oof, [2, 3]) & ~np.isin(DAYS, [8, 9, 11, 12])).sum()),
           "sec": round(time.time() - t), "note": note}
    BOARD.append(row); OOF[(name, cv)] = oof
    display(pd.DataFrame([row]).set_index("step"))
    return f1.mean()

## 4 · Out-of-fold K6 probabilities on both splits

In [22]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
K6 = make_fp(FEATS5C, synth_deg=True)

def oof_proba(fp, groups):
    P = np.zeros((len(train), 5)); t = time.time()
    for g in groups:
        va = np.isin(DAYS, g)
        P[va] = fp(train[~va].reset_index(drop=True), train[va].reset_index(drop=True))
    print(f"  {len(groups)} folds in {time.time() - t:.0f}s"); return P

P = {"LODO": oof_proba(K6, LODO), "BLOCK3": oof_proba(K6, BLOCK3)}
GR = {"LODO": LODO, "BLOCK3": BLOCK3}
def mf1(p): return f1_score(y_all, p.argmax(1), average="macro")
def per_class(p): return dict(zip(LABELS, np.round(f1_score(y_all, p.argmax(1), average=None, labels=range(5)), 3)))
R["K6_plain"] = {cv: round(mf1(P[cv]), 4) for cv in P}
print("K6 plain:", R["K6_plain"])

  13 folds in 73s
  5 folds in 25s
K6 plain: {'LODO': 0.9348, 'BLOCK3': 0.9368}


## 5 · Temporal neighbour smoothing — k sweep on both splits

In [23]:
SPACE = RAW   # the 4 raw signals: closest in all four = closest in time

def smooth_rows(p, X, k):
    if k <= 1: return p
    Z = StandardScaler().fit_transform(X[SPACE])
    _, idx = NearestNeighbors(n_neighbors=min(k, len(Z))).fit(Z).kneighbors(Z)
    return p[idx].mean(axis=1)

def smooth_oof(Pcv, groups, k):
    Q = Pcv.copy()
    for g in groups:                      # smoothing only ever looks inside one held-out set
        va = np.isin(DAYS, g)
        hp = HealthyPlant().fit(train[~va]); f = hp.transform(train[va])
        Q[va] = apply_guard(smooth_rows(Pcv[va], train[va], k), f)
    return Q

KS = [1, 5, 10, 20, 40]
rows = []
for k in KS:
    r = {"k": k}
    for cv in ["LODO", "BLOCK3"]:
        q = smooth_oof(P[cv], GR[cv], k); r[cv] = round(mf1(q), 4)
        if cv == "LODO": r.update({f"L_{l[:5]}": v for l, v in per_class(q).items()})
    rows.append(r)
sweep = pd.DataFrame(rows).set_index("k"); sweep["mean"] = sweep[["LODO", "BLOCK3"]].mean(1).round(4)
display(sweep); R["smoothing_sweep"] = sweep.reset_index().to_dict("records")

best_k = int(sweep.loc[KS[1:], "mean"].idxmax())
use_smooth = bool((sweep.loc[best_k, "LODO"] > sweep.loc[1, "LODO"]) and (sweep.loc[best_k, "BLOCK3"] > sweep.loc[1, "BLOCK3"]))
K_FINAL = best_k if use_smooth else 1
print(f"best k = {best_k}; beats plain K6 on both splits: {use_smooth} -> K_FINAL = {K_FINAL}")
R["K_FINAL"] = K_FINAL

,LODO,L_norma,L_shadi,L_degra,L_short,L_open_,BLOCK3,mean
k,,,,,,,,
1,0.9348,0.947,0.918,0.854,0.960,0.996,0.9368,0.9358
5,0.9404,0.949,0.921,0.868,0.968,0.996,0.9422,0.9413
10,0.9418,0.950,0.921,0.868,0.974,0.996,0.9430,0.9424
20,0.9488,0.950,0.921,0.889,0.988,0.996,0.9516,0.9502
40,0.9514,0.950,0.921,0.903,0.988,0.996,0.9539,0.9526


best k = 40; beats plain K6 on both splits: True -> K_FINAL = 40


## 6 · Diagnostic: shading-vs-normal bias (applied only if ≥ 0.003 gain on both splits)

In [24]:
def bias(p, b):
    q = p.copy(); q[:, L2I["shading"]] *= b; return q
Q = {cv: smooth_oof(P[cv], GR[cv], K_FINAL) for cv in P}
brows = [{"b": b, **{cv: round(mf1(bias(Q[cv], b)), 4) for cv in Q}} for b in [0.8, 0.9, 1.0, 1.1, 1.25]]
bt = pd.DataFrame(brows).set_index("b"); display(bt)
gain = bt.sub(bt.loc[1.0], axis=1)
ok = gain[(gain.LODO >= 0.003) & (gain.BLOCK3 >= 0.003)]
B_FINAL = float(ok.mean(1).idxmax()) if len(ok) else 1.0
print("B_FINAL =", B_FINAL); R["bias_table"] = bt.reset_index().to_dict("records"); R["B_FINAL"] = B_FINAL

final_oof = bias(Q["LODO"], B_FINAL)
R["final_LODO"] = round(mf1(final_oof), 4); R["final_BLOCK3"] = round(mf1(bias(Q["BLOCK3"], B_FINAL)), 4)
R["final_LODO_per_class"] = per_class(final_oof)
cm = confusion_matrix(y_all, final_oof.argmax(1), labels=range(5)); R["final_confusion(rows=true)"] = cm.tolist()
print("final LODO", R["final_LODO"], "BLOCK3", R["final_BLOCK3"], R["final_LODO_per_class"]); print(cm)

,LODO,BLOCK3
b,,
0.80,0.9506,0.9538
0.90,0.9513,0.9538
1.00,0.9514,0.9539
1.10,0.9517,0.9540
1.25,0.9515,0.9537


B_FINAL = 1.0
final LODO 0.9514 BLOCK3 0.9539 {'normal': np.float64(0.95), 'shading': np.float64(0.921), 'degradation': np.float64(0.903), 'short_circuit': np.float64(0.988), 'open_circuit': np.float64(0.996)}
[[25301  1202    84     0     0]
 [ 1348 14903     0     0     0]
 [   43     6   640     0     2]
 [    2     1     3   240     0]
 [    0     0     0     0   242]]


## 7 · Fit K6 on all days, smooth on the test set, write submission

In [25]:
p_test, model = K6.__call__(train, test), None
f_test = HealthyPlant().fit(train).transform(test)
p_s = bias(apply_guard(smooth_rows(p_test, test, K_FINAL), f_test), B_FINAL)
lab_plain = np.array(LABELS)[p_test.argmax(1)]; lab = np.array(LABELS)[p_s.argmax(1)]
out = sub[["id"]].merge(pd.DataFrame({"id": test.id.values, "label": lab}), on="id", how="left")
assert out.label.notna().all() and len(out) == len(sub) and list(out.columns) == list(sub.columns)
fn = f"sub_K6_smooth_k{K_FINAL}_b{B_FINAL}.csv"; out.to_csv(fn, index=False)
sub[["id"]].merge(pd.DataFrame({"id": test.id.values, "label": lab_plain}), on="id", how="left").to_csv("sub_K6.csv", index=False)
R["file"] = fn; R["md5"] = hashlib.md5(open(fn, "rb").read()).hexdigest()
R["test_counts_plain"] = pd.Series(lab_plain).value_counts().reindex(LABELS).fillna(0).astype(int).to_dict()
R["test_counts_final"] = pd.Series(lab).value_counts().reindex(LABELS).fillna(0).astype(int).to_dict()
R["test_rows_changed"] = int((lab != lab_plain).sum())
R["test_changes(from->to)"] = pd.Series([f"{a}->{b}" for a, b in zip(lab_plain, lab) if a != b]).value_counts().head(8).to_dict()
print("wrote", fn, R["md5"]); print(R["test_changes(from->to)"])
R["total_runtime_min"] = round((time.time() - T0) / 60, 1)
print("=====BEGIN SMOOTH RESULTS=====")
for k, v in R.items(): print(f"{k}: {json.dumps(v, default=lambda o: o.item() if hasattr(o, 'item') else str(o), separators=(',', ':'))}")
print("=====END SMOOTH RESULTS=====")

wrote sub_K6_smooth_k40_b1.0.csv de5c2dca08cbd852efb647b173c810de
{'normal->shading': 86, 'shading->normal': 67, 'degradation->shading': 4, 'degradation->short_circuit': 3, 'shading->short_circuit': 2, 'normal->degradation': 2, 'degradation->normal': 1}
=====BEGIN SMOOTH RESULTS=====
K6_plain: {"LODO":0.9348,"BLOCK3":0.9368}
smoothing_sweep: [{"k":1,"LODO":0.9348,"L_norma":0.947,"L_shadi":0.918,"L_degra":0.854,"L_short":0.96,"L_open_":0.996,"BLOCK3":0.9368,"mean":0.9358},{"k":5,"LODO":0.9404,"L_norma":0.949,"L_shadi":0.921,"L_degra":0.868,"L_short":0.968,"L_open_":0.996,"BLOCK3":0.9422,"mean":0.9413},{"k":10,"LODO":0.9418,"L_norma":0.95,"L_shadi":0.921,"L_degra":0.868,"L_short":0.974,"L_open_":0.996,"BLOCK3":0.943,"mean":0.9424},{"k":20,"LODO":0.9488,"L_norma":0.95,"L_shadi":0.921,"L_degra":0.889,"L_short":0.988,"L_open_":0.996,"BLOCK3":0.9516,"mean":0.9502},{"k":40,"LODO":0.9514,"L_norma":0.95,"L_shadi":0.921,"L_degra":0.903,"L_short":0.988,"L_open_":0.996,"BLOCK3":0.9539,"mean":0.952

In [26]:
from IPython.display import FileLink, display
display(FileLink("sub_K6_smooth_k40_b1.0.csv"))

/kaggle/working/sub_K6_smooth_k40_b1.0.csv